# QQQI / QQQ / TQQQ v4.2 baseline experiment suite

Current research baseline: `qqqi_qqq_tqqq_vxn_bridge_v4_2`.  
Cost convention: 10 bps per turnover unit.  
Status: research-only; not trade-ready.

This notebook reviews state-1 lifecycle attribution, tail-risk diagnostics and the two predeclared SGOV defensive-asset challengers. It does not search bridge weights or alter the frozen v4.2 signal trace.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DIAGNOSTICS = ROOT / 'artifacts/evidence/qqqi_qqq_tqqq_v4_2_baseline_diagnostics'
SGOV = ROOT / 'artifacts/evidence/qqqi_qqq_tqqq_v4_2_sgov_defense_v4_3_research'
assert DIAGNOSTICS.exists(), DIAGNOSTICS
assert SGOV.exists(), SGOV


## 1. Baseline and tail-risk comparison

In [ ]:
headline = pd.read_csv(DIAGNOSTICS / 'headline_metrics.csv', index_col=0)
tail = pd.read_csv(DIAGNOSTICS / 'tail_risk_comparison.csv', index_col=0)
display(headline)
display(tail)

## 2. State-1 lifecycle attribution

Each row is one actual contiguous state-1 holding interval. The lifecycle label records the state before and after the bridge, for example `0->1->2` or `0->1->0`.

In [ ]:
episodes = pd.read_csv(DIAGNOSTICS / 'state_1_lifecycle_episodes.csv', parse_dates=['start_date', 'end_date'])
lifecycle = pd.read_csv(DIAGNOSTICS / 'state_1_lifecycle_summary.csv')
display(episodes)
display(lifecycle)
if not episodes.empty:
    ax = episodes.set_index('start_date')['net_return_delta'].mul(100).plot(
        kind='bar', figsize=(13, 4), title='v4.2 minus v4.1 net return by actual state-1 interval (pp)'
    )
    ax.axhline(0, linewidth=1)
    ax.set_ylabel('Percentage points')
    plt.tight_layout()
    plt.show()

## 3. SGOV defensive-asset experiment

Exactly two structural challengers are admitted: pure SGOV defense and a 50/50 QQQI-SGOV defensive reserve. Signals, state 2, execution timing and costs remain unchanged.

In [ ]:
sgov_metrics = pd.read_csv(SGOV / 'headline_metrics.csv', index_col=0)
sgov_chrono = pd.read_csv(SGOV / 'chronological_metrics.csv')
sgov_summary = json.loads((SGOV / 'experiment_summary.json').read_text(encoding='utf-8'))
display(sgov_metrics)
display(sgov_chrono)
display(pd.DataFrame(sgov_summary['diagnostics']['tail_risk']).T)

In [ ]:
equity = {}
for key in ('current_v4_2', 'sgov_pure_defense', 'qqqi_sgov_blended_defense'):
    daily = pd.read_csv(SGOV / f'daily_{key}.csv', parse_dates=['date']).set_index('date')
    equity[key] = daily['equity']
equity = pd.DataFrame(equity)
ax = equity.plot(figsize=(13, 5), title='v4.2 and frozen SGOV challengers: equity')
ax.set_ylabel('Equity')
ax.grid(True, alpha=0.3)
plt.show()
drawdown = equity.div(equity.cummax()).sub(1.0)
ax = drawdown.plot(figsize=(13, 4), title='v4.2 and frozen SGOV challengers: drawdown')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.3)
plt.show()

## 4. Decision discipline

- v4.2 is the current research baseline.
- v4.1 remains the immutable historical signal comparator.
- The SGOV study evaluates defensive architecture, not signal quality.
- No candidate is promoted automatically from this notebook.
- Tail-risk improvement must be weighed against CAGR, Sharpe, Sortino, Calmar and chronological stability.
- Prospective evidence remains mandatory before any trade-ready designation.